In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import Huber
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

import joblib
import optuna

# ────────────────────────
# 데이터 준비
df = pd.read_csv('./merged_all_data.csv')
df['y_sin'] = np.sin(np.deg2rad(df['y_angle']))
df['y_cos'] = np.cos(np.deg2rad(df['y_angle']))
df['distance'] = np.sqrt((df['x_pos'] - df['x_target'])**2 + 
                         (df['y_pos'] - df['y_target'])**2 + 
                         (df['z_pos'] - df['z_target'])**2)
df['dy'] = df['y_pos'] - df['y_target']

X = df[['distance', 'dy']].values
y = df[['y_sin', 'y_cos']].values

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

def to_deg(y_vec):
    return np.rad2deg(np.arctan2(y_vec[:, 0], y_vec[:, 1]))
y_true_angle = to_deg(y_test)

# ────────────────────────
# Optuna 하이퍼파라미터 튜닝 (DNN + XGBoost)
def objective(trial):
    # DNN
    dnn_units = trial.suggest_categorical("dnn_units", [64, 128, 256])
    dnn_lr = trial.suggest_loguniform("dnn_lr", 1e-5, 1e-3)

    dnn_model = Sequential([
        Dense(dnn_units, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(dnn_units, activation='relu'),
        Dense(2)
    ])
    dnn_model.compile(optimizer=Adam(learning_rate=dnn_lr), loss=Huber())
    dnn_model.fit(
        X_train, y_train,
        epochs=150, batch_size=1024, validation_split=0.2,
        callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
        verbose=0
    )
    y_pred_dnn = dnn_model.predict(X_test)

    # XGBoost
    xgb_n_estimators = trial.suggest_int("xgb_n_estimators", 80, 200)
    xgb_max_depth = trial.suggest_int("xgb_max_depth", 2, 6)
    xgb_lr = trial.suggest_loguniform("xgb_lr", 0.01, 0.3)

    xgb_model = MultiOutputRegressor(XGBRegressor(
        n_estimators=xgb_n_estimators,
        max_depth=xgb_max_depth,
        learning_rate=xgb_lr,
        tree_method='hist',
        random_state=42,
        verbosity=0
    ))
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    # 앙상블
    y_pred_ensemble = (y_pred_dnn + y_pred_xgb) / 2
    pred_angle = to_deg(y_pred_ensemble)
    mae = mean_absolute_error(y_true_angle, pred_angle)
    return mae

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print("Best trial:", study.best_trial.params)

# ────────────────────────
# 최적 파라미터로 재학습 후 저장
params = study.best_trial.params

# DNN
dnn_model = Sequential([
    Dense(params['dnn_units'], activation='relu', input_shape=(X_train.shape[1],)),
    Dense(params['dnn_units'], activation='relu'),
    Dense(2)
])
dnn_model.compile(optimizer=Adam(learning_rate=params['dnn_lr']), loss=Huber())
dnn_model.fit(
    X_train, y_train,
    epochs=300, batch_size=1024, validation_split=0.2,
    callbacks=[EarlyStopping(patience=15, restore_best_weights=True)],
    verbose=1
)
dnn_model.save("best_dnn_model.h5")

# XGBoost
xgb_model = MultiOutputRegressor(XGBRegressor(
    n_estimators=params['xgb_n_estimators'],
    max_depth=params['xgb_max_depth'],
    learning_rate=params['xgb_lr'],
    tree_method='hist',
    random_state=42,
    verbosity=0
))
xgb_model.fit(X_train, y_train)
joblib.dump(xgb_model, "best_xgb_model.pkl")

# 변환기 저장
joblib.dump(poly, "poly_transformer.pkl")
joblib.dump(scaler, "scaler.pkl")

# ────────────────────────
# 앙상블 평가 & 시각화
y_pred_dnn = dnn_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)
y_pred_ensemble = (y_pred_dnn + y_pred_xgb) / 2

def evaluate_model(name, y_pred):
    angle = to_deg(y_pred)
    return {
        'Model': name,
        'MAE_y': mean_absolute_error(y_true_angle, angle),
        'RMSE_y': np.sqrt(mean_squared_error(y_true_angle, angle)),
        'R2_y': r2_score(y_true_angle, angle)
    }

results = [
    evaluate_model("DNN", y_pred_dnn),
    evaluate_model("XGBoost", y_pred_xgb),
    evaluate_model("DNN + XGBoost", y_pred_ensemble)
]
results_df = pd.DataFrame(results)
print(results_df)

plt.figure(figsize=(14, 4))
for i, (name, pred) in enumerate(zip(results_df['Model'], [y_pred_dnn, y_pred_xgb, y_pred_ensemble])):
    plt.subplot(1, 3, i+1)
    plt.scatter(y_true_angle, to_deg(pred), alpha=0.4)
    plt.plot([-5, 10], [-5, 10], 'r--')
    plt.title(name)
    plt.xlabel("True y_angle")
    plt.ylabel("Predicted y_angle")
    plt.grid(True)

plt.tight_layout()
plt.suptitle("DNN + XGBoost Ensemble (Optuna Tuned)", fontsize=15, y=1.05)
plt.savefig("ensemble_optuna_y_angle.png", dpi=300)
plt.show()